In [88]:
COUNTRY_ID = "VN"

In [89]:
API_KEYS = ["HG23wq9Qv84ZKe7AYbsKDm8CtQkJuM00","sdTfAAfKz6qRglbYVr9oRAHjgs89A7ce","miAUi2SsXWhP8xENbcsjbihhAd3Ui6u7","03SOqMhpQGGMoG1GLLcTSkKtAnpKxbWA","Xjf1OLB4JwiRgkqlOAPgxqHnv2adG1GI"]

In [90]:
API_KEY_INDEX = 0
API_KEY = API_KEYS[API_KEY_INDEX]

In [109]:
# Cấu hình MongoDB
USER = "admin"
PASS = quote_plus("hungnt121@gmail.com")  # Encode password
HOST = "jenterprise-cluster.50c8w.mongodb.net"
DB_NAME = "JEnterprise"
PROVINCES_TABLE = "provinces"
WEATHER_TABLE = "weathers"

# MONGO_URI = f"mongodb+srv://{USER}:{PASS}@{HOST}/{DB_NAME}?retryWrites=true&w=majority"
MONGO_URI = f"mongodb+srv://{USER}:{PASS}@{HOST}/{DB_NAME}?retryWrites=true&w=majority&appName=JENterprise-Cluster"

# **Thư viện**


In [8]:
!python -m pip install "pymongo[srv]"==3.11
!pip install requests

import requests
import json

import certifi
from pymongo import MongoClient
from urllib.parse import quote_plus

from dateutil import parser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.7/771.7 kB 20.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.4/188.4 kB 10.9 MB/s eta 0:00:00
  Created wheel for pymongo: filename=pymongo-3.11.0-cp311-cp311-linux_x86_64.whl size=495609 sha256=e2ded7e000318db47e08b980bb78ff3b3f7cf741c47e2e7dd8b77354823c0d35
  Stored in directory: /root/.cache/pip/wheels/43/00/27/6d27c275881078538e7cd04e595f2f3a1f14b1ef9e32e40583
Successfully built pymongo


# Data Processing

In [92]:
def extract_date(datetime_str):
    dt = parser.parse(datetime_str)
    return dt.day, dt.month, dt.year

In [93]:
def round_to_half(n):
    return round(n * 2) / 2

In [94]:
def weather_extract_temperature(day_info):
  minTemp = 0
  minUnit = "C"
  maxTemp = 0
  maxUnit = "C"
  try:
    minTemp = day_info.get("Temperature", {}).get("Minimum", {}).get("Value", "32")
    minUnit = day_info.get("Temperature", {}).get("Minimum", {}).get("Unit", "F")
    if minUnit == "F":
      minTemp = round_to_half((minTemp - 32) * 5/9)
      minUnit = "C"

    maxTemp = day_info.get("Temperature", {}).get("Maximum", {}).get("Value", "32")
    maxUnit = day_info.get("Temperature", {}).get("Maximum", {}).get("Unit", "F")
    if maxUnit == "F":
      maxTemp = round_to_half((maxTemp - 32) * 5/9)
      maxUnit = "C"
  except:
    pass
  return {
      "minTemp": minTemp,
      "minUnit": minUnit,
      "maxTemp": maxTemp,
      "maxUnit": maxUnit
  }

In [95]:
def weather_extract_airandpollen(day_info):
  airQualityValue = 0
  airQualityCategory = "Unknown"
  uvIndexValue = 0
  uvIndexCategory = "Unknown"

  # Chất lượng không khí và chỉ số UV
  try:
    airAndPollen = day_info.get("AirAndPollen", [{}])
    for item in airAndPollen:
      if item.get("Name", "") == "AirQuality":
        airQualityValue = item.get("Value", 0)
        airQualityCategory = item.get("Category", "Unknown")
      if item.get("Name", "") == "UVIndex":
        uvIndexValue = item.get("Value", 0)
        uvIndexCategory = item.get("Category", "Unknown")
  except:
    pass
  return {
      "airQualityValue": airQualityValue,
      "airQualityCategory": airQualityCategory,
      "uvIndexValue": uvIndexValue,
      "uvIndexCategory": uvIndexCategory
  }

In [96]:
def weather_extract_windSpeed(time_info):
  # Tốc độ gió
  # 🌬 Nhẹ	1 - 10 mi/h	Gió nhẹ, không ảnh hưởng nhiều
  # 💨 Vừa phải	11 - 20 mi/h	Cảm nhận rõ gió thổi, lá cây rung mạnh
  # 🌪 Mạnh	21 - 40 mi/h	Khó giữ ô, ảnh hưởng xe máy, biển động
  # 🌀 Rất mạnh	41 - 58 mi/h	Cây đổ, nguy hiểm khi di chuyển ngoài trời
  # 🌪 Bão, siêu bão	> 58 mi/h	Có thể gây thiệt hại lớn, nguy hiểm
  # Mạnh ảnh hưởng đến xe máy, đi biển, hoặc các hoạt động ngoài trời.
  windSpeed = 0
  windDirection = "N/A"
  windGustSpeed = 0
  windGustDirection = "N/A"
  try:
    windSpeed = time_info.get("Wind", {}).get("Speed", {}).get("Value", 0)
    windDirection = time_info.get("Wind", {}).get("Direction", {}).get("English", "")
    windGustSpeed = time_info.get("WindGust", {}).get("Speed", {}).get("Value", 0)
    windGustDirection = time_info.get("WindGust", {}).get("Direction", {}).get("English", "")
  except:
    pass
  return windSpeed, windDirection, windGustSpeed, windGustDirection

In [97]:
def weather_extract_humidity(time_info):
  # Độ ẩm
  minHumidity = 0
  maxHumidity = 0
  avgHumidity = 0
  try:
    minHumidity = time_info.get("RelativeHumidity", {}).get("Minimum", 0)
    maxHumidity = time_info.get("RelativeHumidity", {}).get("Maximum", 0)
    avgHumidity = time_info.get("RelativeHumidity", {}).get("Average", 0)
  except:
    pass
  return minHumidity, maxHumidity, avgHumidity

In [98]:
def weather_extract_weatherProbability(time_info):
  precipitationProbability = 0
  thunderstormProbability = 0
  rainProbability = 0
  snowProbability = 0
  iceProbability = 0
  try:
    precipitationProbability = time_info.get("PrecipitationProbability", 0)
    thunderstormProbability = time_info.get("ThunderstormProbability", 0)
    rainProbability = time_info.get("RainProbability", 0)
    snowProbability = time_info.get("SnowProbability", 0)
    iceProbability = time_info.get("IceProbability", 0)
  except:
    pass
  return precipitationProbability, thunderstormProbability, rainProbability, snowProbability, iceProbability

In [99]:
def weather_extract_cloudCover(time_info):
  cloudCover = 0
  try:
    cloudCover = time_info.get("CloudCover", 0)
  except:
    pass
  return cloudCover

In [100]:
def weather_extract_info(time_info):
  iconPhrase = "Unknown"
  shortPhrase = "Unknown"
  longPhrase = "Unknown"
  try:
    iconPhase = time_info.get("IconPhrase", "Unknown")
    shortPhase = time_info.get("ShortPhrase", "Unknown")
    longPhase = time_info.get("LongPhrase", "Unknown")
  except:
    pass
  return iconPhase, shortPhase, longPhase

In [101]:
def weather_extract_dayNightInfo(day_info):
  day_windSpeed = 0
  day_windDirection = "N/A"
  day_windGustSpeed = 0
  day_windGustDirection = "N/A"
  day_minHumidity = 0
  day_maxHumidity = 0
  day_avgHumidity = 0
  day_precipitationProbability = 0
  day_thunderstormProbability = 0
  day_rainProbability = 0
  day_snowProbability = 0
  day_iceProbability = 0
  day_cloudCover = 0
  day_iconPhrase = "Unknown"
  day_shortPhrase = "Unknown"
  day_longPhrase = "Unknown"

  night_windSpeed = 0
  night_windDirection = "N/A"
  night_windGustSpeed = 0
  night_windGustDirection = "N/A"
  night_minHumidity = 0
  night_maxHumidity = 0
  night_avgHumidity = 0
  night_precipitationProbability = 0
  night_thunderstormProbability = 0
  night_rainProbability = 0
  night_snowProbability = 0
  night_iceProbability = 0
  night_cloudCover = 0
  night_iconPhrase = "Unknown"
  night_shortPhrase = "Unknown"
  night_longPhrase = "Unknown"

  try:
    timeInfo = day_info.get("Day", {})
    day_windSpeed, day_windDirection , day_windGustSpeed, day_windGustDirection = weather_extract_windSpeed(timeInfo)
    day_minHumidity, day_maxHumidity, day_avgHumidity = weather_extract_humidity(timeInfo)
    day_precipitationProbability, day_thunderstormProbability, day_rainProbability, day_snowProbability, day_iceProbability = weather_extract_weatherProbability(timeInfo)
    day_cloudCover = weather_extract_cloudCover(timeInfo)
    day_iconPhrase, day_shortPhrase, day_longPhrase = weather_extract_info(timeInfo)
  except:
    pass

  try:
    timeInfo = day_info.get("Night", {})
    night_windSpeed, night_windDirection , night_windGustSpeed, night_windGustDirection = weather_extract_windSpeed(timeInfo)
    night_minHumidity, night_maxHumidity, night_avgHumidity = weather_extract_humidity(timeInfo)
    night_precipitationProbability, night_thunderstormProbability, night_rainProbability, night_snowProbability, night_iceProbability = weather_extract_weatherProbability(timeInfo)
    night_cloudCover = weather_extract_cloudCover(timeInfo)
    night_iconPhrase, night_shortPhrase, night_longPhrase = weather_extract_info(timeInfo)
  except:
    pass

  return {
      "dayWindSpeed": day_windSpeed,
      "dayWindDirection": day_windDirection,
      "dayWindGustSpeed": day_windGustSpeed,
      "dayWindGustDirection": day_windGustDirection,
      "dayMinHumidity": day_minHumidity,
      "dayMaxHumidity": day_maxHumidity,
      "dayAvgHumidity": day_avgHumidity,
      "dayPrecipitationProbability": day_precipitationProbability,
      "dayThunderstormProbability": day_thunderstormProbability,
      "dayRainProbability": day_rainProbability,
      "daySnowProbability": day_snowProbability,
      "dayIceProbability": day_iceProbability,
      "dayCloudCover": day_cloudCover,
      "dayIconPhrase": day_iconPhrase,
      "dayShortPhrase": day_shortPhrase,
      "dayLongPhrase": day_longPhrase,

      "nightWindSpeed": night_windSpeed,
      "nightWindDirection": night_windDirection,
      "nightWindGustSpeed": night_windGustSpeed,
      "nightWindGustDirection": night_windGustDirection,
      "nightMinHumidity": night_minHumidity,
      "nightMaxHumidity": night_maxHumidity,
      "nightAvgHumidity": night_avgHumidity,
      "nightPrecipitationProbability": night_precipitationProbability,
      "nightThunderstormProbability": night_thunderstormProbability,
      "nightRainProbability": night_rainProbability,
      "nightSnowProbability": night_snowProbability,
      "nightIceProbability": night_iceProbability,
      "nightCloudCover": night_cloudCover,
      "nightIconPhrase": night_iconPhrase,
      "nightShortPhrase": night_shortPhrase,
      "nightLongPhrase": night_longPhrase
  }

In [133]:
def weatherInfoToMongoItem(weather_info, area_id):
  infos = []
  for day_info in weather_info["DailyForecasts"]:
    try:
      tmpDate = extract_date(day_info["Date"])
      tmpDay, tmpMonth, tmpYear = tmpDate
    except:
      tmpDay, tmpMonth, tmpYear = (1,1,2025)

    temperatureInfo = weather_extract_temperature(day_info)
    airQualityInfo = weather_extract_airandpollen(day_info)
    weatherInfo = weather_extract_dayNightInfo(day_info)

    info = {
      "area_id": area_id,
      "day": tmpDay,
      "month": tmpMonth,
      "year": tmpYear,
      "date": f"{tmpDay}/{tmpMonth}/{tmpYear}",
      "min_temp": temperatureInfo.get("minTemp", 0),
      "max_temp": temperatureInfo.get("maxTemp", 0),
      "air_quality_value": airQualityInfo.get("airQualityValue", 0),
      "air_quality_category": airQualityInfo.get("airQualityCategory", "Unknown"),
      "uv_index_value": airQualityInfo.get("uvIndexValue", 0),
      "uv_index_category": airQualityInfo.get("uvIndexCategory", "Unknown"),
      ## Day Info
      "wind_day_speed": weatherInfo.get("dayWindSpeed", 0),
      "wind_day_dir": weatherInfo.get("dayWindDirection", "Unknown"),
      "wind_day_gust_speed": weatherInfo.get("dayWindGustSpeed", 0),
      "wind_day_gust_dir": weatherInfo.get("dayWindGustDirection", "Unknown"),
      "hum_day_min": weatherInfo.get("dayMinHumidity", 0),
      "hum_day_max": weatherInfo.get("dayMaxHumidity", 0),
      "hum_day_avg": weatherInfo.get("dayAvgHumidity", 0),
      "precip_day_prob": weatherInfo.get("dayPrecipitationProbability", 0),
      "storm_day_prob": weatherInfo.get("dayThunderstormProbability", 0),
      "rain_day_prob": weatherInfo.get("dayRainProbability", 0),
      "snow_day_prob": weatherInfo.get("daySnowProbability", 0),
      "ice_day_prob": weatherInfo.get("dayIceProbability", 0),
      "cloud_day_cover": weatherInfo.get("dayCloudCover", 0),
      "icon_day_phrase": weatherInfo.get("dayIconPhrase", "Unknown"),
      "short_day_phrase": weatherInfo.get("dayShortPhrase", "Unknown"),
      "long_day_phrase": weatherInfo.get("dayLongPhrase", "Unknown"),
      ## Night Info
      "wind_night_speed": weatherInfo.get("nightWindSpeed", 0),
      "wind_night_dir": weatherInfo.get("nightWindDirection", "Unknown"),
      "wind_night_gust_speed": weatherInfo.get("nightWindGustSpeed", 0),
      "wind_night_gust_dir": weatherInfo.get("nightWindGustDirection", "Unknown"),
      "hum_night_min": weatherInfo.get("nightMinHumidity", 0),
      "hum_night_max": weatherInfo.get("nightMaxHumidity", 0),
      "hum_night_avg": weatherInfo.get("nightAvgHumidity", 0),
      "precip_night_prob": weatherInfo.get("nightPrecipitationProbability", 0),
      "storm_night_prob": weatherInfo.get("nightThunderstormProbability", 0),
      "rain_night_prob": weatherInfo.get("nightRainProbability", 0),
      "snow_night_prob": weatherInfo.get("nightSnowProbability", 0),
      "ice_night_prob": weatherInfo.get("nightIceProbability", 0),
      "cloud_night_cover": weatherInfo.get("nightCloudCover", 0),
      "icon_night_phrase": weatherInfo.get("nightIconPhrase", "Unknown"),
      "short_night_phrase": weatherInfo.get("nightShortPhrase", "Unknown"),
      "long_night_phrase": weatherInfo.get("nightLongPhrase", "Unknown"),
    }
    # print(info)
    infos.append(info)
  return infos

# **AccuWeather API**

##### Lấy danh sách dựa trên COUNTRY_ID

In [129]:
def getAreasInfo():
  """
  Fetches administrative areas of Vietnam from the AccuWeather API.

  Args:
      apikey (str): AccuWeather API key.

  Returns:
      list | str: A list of administrative areas or an error message.
  """
  global API_KEY
  global API_KEY_INDEX

  url = f"http://dataservice.accuweather.com/locations/v1/adminareas/{COUNTRY_ID}"
  params = {
      "apikey": API_KEY
  }

  try:
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    result = []
    for item in data:
      english_name = item.get("EnglishName", "")
      search_name = missingSearchName.get(english_name, english_name)

      province_info = {
          "ID": item.get("ID", ""),
          "CountryID": item.get("CountryID", ""),
          "type": item.get("LocalizedType") or item.get("EnglishType") or "Unknown",
          "admin_localName": item.get("LocalizedName", ""),
          "admin_englishName": english_name,
          "searchName": search_name,
      }
      result.append(province_info)
    return result
  except requests.exceptions.RequestException as e:
    print("Error: API rate limit exceeded.")
    API_KEY_INDEX = API_KEY_INDEX + 1
    if API_KEY_INDEX >= len(API_KEYS):
      return []
    else:
      API_KEY = API_KEYS[API_KEY_INDEX]
    return getAreasInfo()

##### Lấy thông tin từng tỉnh/thành với COUNTRY_ID và searchName

In [128]:
def getAreaInfo(province):
  """
   Fetches area details including [latitude, longitude, and location key,...]
    from the AccuWeather API based on the city name.

    Args:
        apikey (str): AccuWeather API key.
        city_name (str): Name of the city to search for.

    Returns:
        dict | str: A dictionary containing 'latitude', 'longitude', and 'key',
                    or an error message.
  """
  global API_KEY
  global API_KEY_INDEX

  url = "http://dataservice.accuweather.com/locations/v1/cities/search"
  params = {
      "apikey": API_KEY,
      "q": province
  }

  countryDetail = {
      "lat": "",
      "lon": "",
      "key": "",
      "localName":"",
      "englishName":""
  }

  try:
    response = requests.get(url, params=params)
    response.raise_for_status()

    json_response = response.json()
    status_code = response.status_code
    if not json_response:
      print("No data found for the specified city.")
    else:
      # Mảng, có thể có nhiều quốc gia. Lấy của Việt Nam thôi -> Country = "VN"
      location = next((loc for loc in json_response if loc["Country"]["ID"] == COUNTRY_ID), None)

      if location:
        try:
          countryDetail["lat"] = location["GeoPosition"]["Latitude"]
          countryDetail["lon"] = location["GeoPosition"]["Longitude"]
          countryDetail["key"] = location["Key"]
          countryDetail["localName"] = location["LocalizedName"]
          countryDetail["englishName"] = location["EnglishName"]
        except:
          pass
      else:
        print("No data found for the specified city.")

    return countryDetail
  except requests.exceptions.RequestException as e:
    print("Error: API rate limit exceeded.")
    API_KEY_INDEX = API_KEY_INDEX + 1
    if API_KEY_INDEX >= len(API_KEYS):
      return []
    else:
      API_KEY = API_KEYS[API_KEY_INDEX]
    return getAreaInfo(province)

##### Các địa điểm chưa đặt tên ứng với adminareas_LocalzedName

**Admin tự cập nhật khi cảm thấy có sự thay đổi**

In [105]:
missingSearchName = dict({
    'Nghệ An': 'Vinh',
    'Quảng Bình': 'Dong Hoi',
    'Quảng Trị': 'Quang Tri',
    'Thừa Thiên-Huế': 'Hue',
    'Quảng Nam': 'Tam Ky',
    'Gia Lai': 'Pleiku',
    'Bình Định': 'Dinh Binh',
    'Phú Yên': 'Tuy Hoa',
    'Đắk Lắk': 'Buon Ma Thuot',
    'Khánh Hòa': 'Nha Trang',
    'Lâm Đồng': 'Da Lat',
    'Ninh Thuận': 'Phan Rang',
    'Đồng Nai': 'Bien Hoa',
    'Bình Thuận': 'Phan Thiet',
    'Bà Rịa - Vũng Tàu': 'Ba Ria',
    'Đồng Tháp': 'Cao Lanh',
    'Tiền Giang': 'My Tho',
    'Kiến Giang': 'Rach Gia',
    'Bình Dương': 'Thu Dau Mot',
    'Bình Phước': 'Dong Xoai',
    'Hà Nam': 'Phu Ly',
    'Vĩnh Phúc': 'Vinh Yen',
    'Đắk Nông': 'Gia Nghia',
    'Hậu Giang': 'Vi Thanh'
})

#### Lấy thời tiết theo key

In [130]:
def getArea5DayWeather(area_key):
  """
   Fetches area 5-day weathers from the AccuWeather API based on the area_key.

    Args:
        apikey (str): AccuWeather API key.
        area_key (str): Key of the area to search for.

    Returns:
        list: A list of dictionaries containing weather information for each day.
  """
  global API_KEY
  global API_KEY_INDEX

  url = f"http://dataservice.accuweather.com/forecasts/v1/daily/5day/{area_key}"
  params = {
      "apikey": API_KEY,
      "details": True
  }

  try:
    response = requests.get(url, params=params)
    response.raise_for_status()

    json_response = response.json()
    status_code = response.status_code
    if not json_response:
      print("No data found for the specified city.")

    return json_response
  except requests.exceptions.RequestException as e:
    print("Error: API rate limit exceeded.")
    API_KEY_INDEX = API_KEY_INDEX + 1
    if API_KEY_INDEX >= len(API_KEYS):
      return []
    else:
      API_KEY = API_KEYS[API_KEY_INDEX]
    return getArea5DayWeather(area_key)

# **MongoDB Controller**

##### Save to MongoDB

In [107]:
def save_to_mongodb(data, mongo_uri, db_name, collection_name):
    """
    Lưu dữ liệu vào MongoDB một cách an toàn.
    """
    if not data:
        print("Không có dữ liệu để lưu.")
        return

    try:
        client = MongoClient(MONGO_URI, tls=True, tlsAllowInvalidCertificates=True)
        # Kết nối database
        db = client[db_name]
        print(f"✅ Đã chọn database: {db_name}")

        # Kiểm tra collection có tồn tại không
        collection_list = db.list_collection_names()
        print("🔍 Danh sách collection:", collection_list)

        # Kết nối collection
        collection = db[collection_name]
        print(f"✅ Đã chọn collection: {collection_name}")

        # Lưu dữ liệu (cập nhật nếu đã có)
        for area in data:
            collection.update_one({"ID": area["ID"]}, {"$set": area}, upsert=True)

        print("✅ Dữ liệu đã được lưu vào MongoDB.")
    except Exception as e:
        print(f"❌ Lỗi kết nối MongoDB: {e}")

In [122]:
def saveWeather_to_mongodb(data):
    """
    Lưu dữ liệu vào MongoDB một cách an toàn.
    """
    if not data:
        print("Không có dữ liệu để lưu.")
        return
    global MONGO_URI,DB_NAME,WEATHER_TABLE

    try:
        client = MongoClient(MONGO_URI, tls=True, tlsAllowInvalidCertificates=True)
        # Kết nối database
        db = client[DB_NAME]

        # Kết nối collection
        collection = db[WEATHER_TABLE]

        # Lưu dữ liệu (cập nhật nếu đã có)
        for weather in data:
            collection.update_one({"ID": f"{weather['area_id']}{weather['date']}"}, {"$set": weather}, upsert=True)

        print("✅ Dữ liệu đã được lưu vào MongoDB.")
    except Exception as e:
        print(f"❌ Lỗi kết nối MongoDB: {e}")

##### Get From MongoDB

In [108]:
from collections.abc import Collection
def getProvinces_mongodb(mongo_uri, db_name, collection_name):
    try:
        client = MongoClient(MONGO_URI, tls=True, tlsAllowInvalidCertificates=True)
        db = client[db_name]
        collection = db[collection_name]
        provinces = []
        collection.find()
        for province in collection.find():
            provinces.append(province)
        return provinces
    except Exception as e:
        print(f"❌ Lỗi kết nối MongoDB: {e}")
        return []

# **Triển khai**

#### Khởi tạo + Kiểm tra

In [110]:
vietnam_areas = getProvinces_mongodb(MONGO_URI, DB_NAME, PROVINCES_TABLE)
provincesEmpty = vietnam_areas.count == 0  or vietnam_areas == []
def updateGlobal():
  global vietnam_areas
  global provincesEmpty
  vietnam_areas = getProvinces_mongodb(MONGO_URI, DB_NAME, PROVINCES_TABLE)
  provincesEmpty = vietnam_areas.count == 0  or vietnam_areas == []

#### Lấy thông tin thành phố và lưu vào MongoDB

In [111]:
if provincesEmpty:
  # Lấy danh sách tỉnh/thành từ AccuWeather
  new_vietnam_areas = getAreasInfo()

  # Get more info of country
  for area in new_vietnam_areas:
    try:
      area_info = getAreaInfo(area['searchName'])
      area['lat'] = f"{area_info['lat']}"
      area['lon'] = f"{area_info['lon']}"
      area['key'] = f"{area_info['key']}"
      area['localName'] = f"{area_info['localName']}"
      area['englishName'] = f"{area_info['englishName']}"
    except:
      pass

  # Lưu vào MongoDB
  if new_vietnam_areas:
    save_to_mongodb(new_vietnam_areas, MONGO_URI, DB_NAME, PROVINCES_TABLE)
  else:
    print(f"Không tìm thấy tỉnh/thành nào thuộc {COUNTRY_ID}")
else:
  hasUpdate = False
  for area in vietnam_areas:
    try:
      if area['lat'] == "" or area['lon'] == "" or area['key'] == "" or area['localName'] == "" or area['englishName'] == "":
        area_info = getAreaInfo(API_KEY, area['searchName'])
        area['lat'] = f"{area_info['lat']}"
        area['lon'] = f"{area_info['lon']}"
        area['key'] = f"{area_info['key']}"
        area['localName'] = f"{area_info['localName']}"
        area['englishName'] = f"{area_info['englishName']}"
        if hasUpdate == False:
          hasUpdate = True
    except:
      print("Lỗi khi lấy thông tin")

  if hasUpdate:
    save_to_mongodb(vietnam_areas, MONGO_URI, DB_NAME, PROVINCES_TABLE)
    print(f"Đã cập nhật thông tin trong database {DB_NAME}.{PROVINCES_TABLE}")
  else:
    print("Không có thông tin cần cập nhật")
updateGlobal()

Không có thông tin cần cập nhật


#### Lấy thông tin thời tiết theo key

In [112]:
area_keys = []
for area in vietnam_areas:
  if area["key"] != "":
    area_keys.append(area["key"])


keys_size = len(area_keys)
if keys_size > 0:
  print(f"{keys_size} key hợp lệ")
else:
  print("Không có key nào hợp lệ")

63 key hợp lệ


In [131]:
weather_data = []
for area_key in area_keys:
  try:
    weather = getArea5DayWeather(area_key)
    infos = weatherInfoToMongoItem(weather,area)
    for info in infos:
      weather_data.append(info)
  except:
    pass

In [134]:
saveWeather_to_mongodb(weather_data)

✅ Dữ liệu đã được lưu vào MongoDB.
